In [13]:
# Зареждане на набора от данни
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

In [24]:
# Конструиране на GAE с GCN енкодер
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE

class GCNEncoder(torch.nn.Module):

    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, 64)
        self.conv2 = GCNConv(64, 32)

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)

        return x

In [25]:
# Създаване на модела
encoder = GCNEncoder(dataset.num_features)
model = GAE(encoder)

In [34]:
# Конструиране на GAE с GraphSAGE енкодер
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, GAE

class GraphSAGEEncoder(torch.nn.Module):

    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, 64, aggr="mean")
        self.conv2 = SAGEConv(64, 32, aggr="mean")

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)

        return x

In [35]:
# Създаване на модела
encoder = GraphSAGEEncoder(dataset.num_features)
model = GAE(encoder)

In [36]:
# Инициализиране на оптимизатора
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01)

In [37]:
# Създаване на тренировъчен и тестов набор от данни
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.10,
    is_undirected=True,
    add_negative_train_samples=True)

train_data, val_data, test_data = transform(data)

In [38]:
# Обучение
import time

def train():
    model.train()
    optimizer.zero_grad()
    z = model.encode(
        train_data.x,
        train_data.edge_index)
    loss = model.recon_loss(
        z,
        train_data.edge_label_index)
    loss.backward()
    optimizer.step()

    return loss.item()

start = time.time()
for epoch in range(1, 201):
    loss = train()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}   Loss: {loss:.4f}")

training_time = time.time() - start

Epoch  20   Loss: 1.2649
Epoch  40   Loss: 1.1844
Epoch  60   Loss: 1.1118
Epoch  80   Loss: 1.0898
Epoch 100   Loss: 1.0713
Epoch 120   Loss: 1.0522
Epoch 140   Loss: 1.0591
Epoch 160   Loss: 1.0397
Epoch 180   Loss: 1.0462
Epoch 200   Loss: 1.0244


In [39]:
# Оценяване
model.eval()

with torch.no_grad():
    z = model.encode(
        test_data.x,
        test_data.edge_index
    )

reconstruction_loss = model.recon_loss(
    z,
    test_data.edge_label_index
).item()

pos_edge_index = test_data.edge_label_index[
    :, test_data.edge_label == 1
]

neg_edge_index = test_data.edge_label_index[
    :, test_data.edge_label == 0]

auc, ap = model.test(
    z,
    pos_edge_index,
    neg_edge_index)

In [40]:
# Извеждане на резултатите
print(f"Reconstruction Loss: {reconstruction_loss:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Average Precision: {ap:.4f}")
print(f"Training time: {training_time:.2f} s")
print(f"Embeddings shape: {z.shape}")

Reconstruction Loss: 1.3753
AUC: 0.8191
Average Precision: 0.8317
Training time: 13.77 s
Embeddings shape: torch.Size([2708, 32])


In [41]:
# Използване на модела за прогнозиране на връзки
model.eval()

with torch.no_grad():
    z = model.encode(data.x, data.edge_index)

edge = torch.tensor([[10],
                     [25]])

prob = model.decoder(
    z,
    edge,
    sigmoid=True
)

print(f"Probability: {prob.item():.4f}")

Probability: 0.5140


In [42]:
model

GAE(
  (encoder): GraphSAGEEncoder(
    (conv1): SAGEConv(1433, 64, aggr=mean)
    (conv2): SAGEConv(64, 32, aggr=mean)
  )
  (decoder): InnerProductDecoder()
)

In [43]:
# Брой параметри
num_params = sum(p.numel() for p in model.parameters())
print(num_params)

187616
